# 8.8 - Conversational RAG: Memory + Retrieval

**Module 8 - RAG Systems** | Netsetos GenAI Engineering

This notebook turns single-turn RAG into a chatbot by adding two moves to the pipeline: **reformulate** a follow-up into a standalone question before retrieval, and **remember** the turn after generation. You build the whole loop by hand with the google-genai SDK, then map it onto the 2026 stack - LangGraph, LlamaIndex, six memory architectures, production hardening, multi-turn evaluation, and India's language + compliance realities.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Uncomment the install line on Colab or a fresh environment. It pulls every library the notebook touches across all ten steps - the by-hand google-genai SDK plus the framework stacks (LangGraph, LlamaIndex, DeepEval) you only meet later.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install numpy sentence-transformers langchain langgraph llama-index llama-index-llms-google-genai llama-index-embeddings-google-genai deepeval google-genai python-dotenv -q  # noqa


**Explanation**

One-time environment prep - installs the full dependency set so no later cell fails on a missing import.

**How the code works - step by step**
- **numpy** - vectors and cosine-similarity math for the by-hand retriever.
- **sentence-transformers** - the BGE-m3 multilingual embedder used in the India step.
- **langchain / langgraph** - the 2026 agent + checkpointer memory stack.
- **llama-index** (+ google-genai llm/embedding plugins) - the one-line ChatEngine.
- **deepeval** - multi-turn conversation evaluation.
- **google-genai** - the current Gemini SDK every by-hand step calls.
- **python-dotenv** - loads keys from a `.env` file.

**In one line:** install once, run every step.

**What you'll see:** (no output - environment prep)

## Setup - load your API key

Every by-hand step calls Gemini, so the SDK needs a key on the environment. This cell reads it (or a placeholder) without ever hardcoding a secret - the recommended pattern for any notebook that ships.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY` from aistudio.google.com).

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

Key-loading boilerplate, not a model call - it makes `genai.Client()` in later cells find its credential from the environment.

**How the code works - step by step**
- **`os.environ.setdefault("GOOGLE_API_KEY", "")`** - registers the key name if it is not already set, leaving any real value from your shell or `.env` untouched.
- The empty-string default means the cell runs cleanly even before you paste a key - the actual model calls are what will fail without one.

**In one line:** wire the Gemini key into the environment so `genai.Client()` picks it up.

**What you'll see:** (no output - environment prep)

## 1 - Query reformulation, the missing piece

Retrieval embeds whatever string you hand it, so a follow-up like "how much does *it* cost?" retrieves garbage - there is no chunk about "it." Reformulation is the single move that fixes this: use the LLM to rewrite the follow-up into a standalone question, resolving every pronoun against the recent history, BEFORE you retrieve. This is the one change that turns a Q&A box into a chatbot.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`).

In [ ]:
# QUERY REFORMULATION - resolve pronouns + context into a standalone question
# pip install google-genai
from google import genai

client = genai.Client()   # reads GOOGLE_API_KEY from the environment

class QueryReformulator:
    def reformulate(self, follow_up, history):
        if not history:
            return follow_up          # first turn: nothing to resolve
        hist = "\n".join(f"{m['role']}: {m['content']}" for m in history[-6:])
        prompt = (
            "Rewrite the follow-up as a STANDALONE question using the history.\n"
            "Resolve every pronoun (it, that, they, this). Return ONLY the question.\n\n"
            f"History:\n{hist}\n\nFollow-up: {follow_up}\n\nStandalone question:"
        )
        return client.models.generate_content(
            model="gemini-2.5-flash", contents=prompt).text.strip()

# Demo: same history, three follow-ups that lean on it
reformulator = QueryReformulator()
history = [
    {"role": "user", "content": "What courses does Netsetos offer?"},
    {"role": "assistant", "content": "Netsetos offers GenAI, Data Science, Cloud, and Full-Stack courses."},
]
for f in ["How much does it cost?", "Can I get a refund on that?", "What are the prerequisites?"]:
    print(f"  follow-up:   {f!r}")
    print(f"  standalone:  {reformulator.reformulate(f, history)!r}\n")

# Output:
#   follow-up:   'How much does it cost?'
#   standalone:  'How much does the Netsetos GenAI course cost?'
#   follow-up:   'Can I get a refund on that?'
#   standalone:  'What is the refund policy for the Netsetos GenAI course?'
#   follow-up:   'What are the prerequisites?'   (already standalone, returned ~as-is)

**Explanation**

A tiny one-method class that does exactly one thing: ask the LLM to spell out a pronoun-laden follow-up into a self-contained question. It is a preprocessing step, not the retriever itself - the rewritten string is what you embed; the original is what you show the user.

**How the code works - step by step**
- **`genai.Client()`** - reads the key from the environment; created once, reused for every call.
- **`reformulate(follow_up, history)`** - the whole feature. On the first turn (`not history`) it returns the message unchanged - nothing to resolve.
- Otherwise it flattens the last six history messages into a `role: content` block and builds a prompt that instructs the model to resolve `it / that / they / this` and return ONLY the standalone question.
- **`generate_content(model="gemini-2.5-flash", ...)`** - runs the rewrite; `.text.strip()` cleans the result.
- The demo replays one fixed history against three follow-ups of decreasing pronoun-dependence.

**In one line:** flatten recent history -> ask the LLM to resolve the pronoun -> return a clean standalone question.

**What you'll see:** For each follow-up it prints the raw string and its rewrite: "How much does it cost?" becomes "How much does the Netsetos GenAI course cost?", "Can I get a refund on that?" becomes a refund-policy question, and the already-standalone "What are the prerequisites?" returns roughly as-is.

## 2 - The full conversational RAG loop

Step 1 fixed retrieval; now wire it into the whole loop so state actually accumulates. This class is the smallest thing that deserves the name chatbot: it embeds the documents once, then on each turn reformulates, retrieves on the standalone query, generates from context plus recent history, and appends the exchange to memory. Every later framework is this exact loop with the parts swapped for industrial components.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`).

In [ ]:
# CONVERSATIONAL RAG - the complete reformulate->retrieve->generate->remember loop
# pip install google-genai numpy
import numpy as np
from google import genai

client = genai.Client()

def embed(text):
    r = client.models.embed_content(model="gemini-embedding-001", contents=text)
    return np.array(r.embeddings[0].values)

class ConversationalRAG:
    def __init__(self, docs, max_history=10):
        self.history, self.max_history = [], max_history
        self.chunks = docs
        self.embs = [embed(d["text"]) for d in docs]   # index once

    def _reformulate(self, query):
        if not self.history: return query
        hist = "\n".join(f"{m['role']}: {m['content'][:100]}" for m in self.history[-6:])
        return client.models.generate_content(model="gemini-2.5-flash",
            contents=f"Rewrite as a standalone question, resolving pronouns.\n{hist}\nFollow-up: {query}\nStandalone:").text.strip()

    def _retrieve(self, query, k=3):
        qe = embed(query)
        order = sorted(range(len(self.chunks)), key=lambda i: -float(np.dot(qe, self.embs[i])))
        return [self.chunks[i] for i in order[:k]]   # rank by index: O(n log n), duplicate-safe

    def chat(self, user_msg):
        standalone = self._reformulate(user_msg)                 # 1. reformulate
        docs = self._retrieve(standalone)                        # 2. retrieve on the clean query
        context = "\n".join(d["text"] for d in docs)
        hist = "\n".join(f"{m['role']}: {m['content'][:80]}" for m in self.history[-4:])
        answer = client.models.generate_content(model="gemini-2.5-flash",  # 3. generate
            contents=f"Answer from context only, conversationally.\n\nContext:\n{context}\n\n{hist}\nUser: {user_msg}\nAssistant:").text.strip()
        self.history += [{"role":"user","content":user_msg},        # 4. remember
                         {"role":"assistant","content":answer}]
        self.history = self.history[-self.max_history*2:]
        return {"answer": answer, "standalone": standalone,
                "sources": [d.get("source","?") for d in docs]}

docs = [
    {"text":"Netsetos offers GenAI, Data Science, Cloud, and Full-Stack courses.","source":"about"},
    {"text":"GenAI course costs 14999 rupees: lifetime access, Discord, weekly live sessions.","source":"pricing"},
    {"text":"Refund policy: full refund within 7 days, 50% for 8-30 days, none after 30.","source":"refund"},
    {"text":"Prerequisites: basic Python and high-school math. No ML experience needed.","source":"prereqs"},
]
bot = ConversationalRAG(docs)
for msg in ["What courses does Netsetos offer?", "How much does the GenAI one cost?",
            "What if I don't like it?", "How long do I have for that?"]:
    r = bot.chat(msg)
    print(f"You: {msg}\nBot: {r['answer'][:90]}\n  [standalone: {r['standalone'][:55]!r} | {r['sources']}]\n")

# Output:
# You: How much does the GenAI one cost?
# Bot: The Netsetos GenAI course costs 14,999 rupees ...
#   [standalone: 'How much does the Netsetos GenAI course cost?' | ['pricing']]
# You: What if I don't like it?
#   [standalone: 'What is the refund policy for the GenAI course?' | ['refund']]

**Explanation**

One class that closes the reformulate -> retrieve -> generate -> remember loop. Read it as: index the docs once at construction, then let each `chat` call thread the four moves and grow the history, capped crudely so the prompt stays bounded.

**How the code works - step by step**
- **`embed(text)`** - one call to `gemini-embedding-001`, returned as a numpy array for dot products.
- **`__init__`** - stores the chunks and embeds each one once (`self.embs`), so retrieval never re-embeds the corpus.
- **`_reformulate`** - step 1 inlined: skip on the first turn, else rewrite the follow-up against the last six turns.
- **`_retrieve`** - embeds the standalone query and sorts chunk indices by descending cosine similarity, returning the top `k` (ranking by index is O(n log n) and duplicate-safe).
- **`chat`** - runs the four numbered moves in order and returns the answer plus the `standalone` rewrite and the `sources` used.
- **`self.history = self.history[-max_history*2:]`** - the crude memory cap: keep the last N exchanges verbatim, silently drop older ones.

**In one line:** index once, then each turn reformulate -> retrieve -> generate -> append, truncating old history.

**What you'll see:** Four progressively vaguer follow-ups, each printed with its answer, the standalone rewrite, and the source doc it hit: "the GenAI one" routes to `pricing`, "what if I don't like it?" routes to `refund` - turns that would all fail retrieval on their raw pronouns.

## 3 - Memory management: sliding window vs summarization

Real conversations hit 50+ turns and you cannot send them all. The crude cap from step 2 throws away context the moment things get interesting, so production picks HOW to forget. This cell builds the two foundational strategies - keep the last N turns verbatim (window), or compress the overflow into a running LLM summary (summarize) - the atoms every framework's memory is built from.

> **Needs a Gemini API key** for the summarize strategy (set `GOOGLE_API_KEY`).

In [ ]:
# MEMORY MANAGEMENT - sliding window (fast, lossy) vs summarization (smart, costs a call)
# pip install google-genai
from google import genai

client = genai.Client()

class ChatMemory:
    def __init__(self, strategy="window", window_size=6):
        self.strategy, self.window_size = strategy, window_size
        self.messages, self.summary = [], ""

    def add(self, role, content):
        self.messages.append({"role": role, "content": content})

    def get_context(self):
        if self.strategy == "window":
            return self.messages[-self.window_size:]          # keep the last N, drop the rest
        # summarize: condense everything older than the window into a running summary
        if len(self.messages) > self.window_size:
            old = self.messages[:-self.window_size]
            text = "\n".join(f"{m['role']}: {m['content'][:80]}" for m in old)
            self.summary = client.models.generate_content(model="gemini-2.5-flash",
                contents=f"Summarize this conversation in 2-3 sentences:\n{text}").text.strip()
            self.messages = self.messages[-self.window_size:]
        ctx = list(self.messages)
        if self.summary:
            ctx.insert(0, {"role":"system", "content":f"Earlier: {self.summary}"})
        return ctx

for strat in ["window", "summarize"]:
    mem = ChatMemory(strategy=strat, window_size=4)
    for i in range(8):
        mem.add("user", f"Question {i+1}"); mem.add("assistant", f"Answer {i+1}")
    ctx = mem.get_context()
    print(f"{strat:10s}: {len(ctx)} messages in context"
          + (f" | summary: {mem.summary[:50]}..." if mem.summary else ""))

# Output:
# window    : 4 messages in context
# summarize : 5 messages in context | summary: The user asked several questions about ...

**Explanation**

One `ChatMemory` class with a strategy switch, so you can watch the same 16-message conversation collapse two different ways. It is a comparison harness: window costs nothing and forgets old detail; summarize spends an LLM call to keep the gist.

**How the code works - step by step**
- **`add(role, content)`** - appends a message to the flat list.
- **`get_context()` / window branch** - returns just the last `window_size` messages, dropping the rest. O(1), lossy on old turns.
- **`get_context()` / summarize branch** - when history exceeds the window, it condenses everything older than the window into `self.summary` via one `gemini-2.5-flash` call, trims `self.messages` to the recent window, and prepends the summary as a `system` note.
- The loop feeds 8 Q&A pairs (16 messages) into each strategy with `window_size=4` and prints how many messages survive in context.

**In one line:** window keeps the last N verbatim; summarize keeps the last N plus a one-line digest of the rest.

**What you'll see:** Two lines: `window` keeps 4 messages in context; `summarize` keeps 5 - the last 4 verbatim plus one summary line ("The user asked several questions about ...").

## 4 - The production chatbot: everything together

Steps 1-3 were parts on a workbench; this bolts them into one object with the scaffolding a real deployment needs - a system prompt to shape tone and grounding, source tracking so answers are attributable, and per-turn latency stats so you can see reformulation's cost. Still under 40 lines, and it is the mental model you carry into the framework steps.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`).

In [ ]:
# PRODUCTION CONVERSATIONAL RAG - system prompt + sources + latency stats
# pip install google-genai numpy
import numpy as np, time
from google import genai

client = genai.Client()
def embed(t): return np.array(client.models.embed_content(model="gemini-embedding-001", contents=t).embeddings[0].values)

class ProductionChatbot:
    def __init__(self, docs, system="You are a Netsetos support assistant. Answer from context only."):
        self.system, self.history = system, []
        self.chunks, self.embs = docs, [embed(d["text"]) for d in docs]
        self.stats = {"turns":0, "reformulated":0}

    def chat(self, msg):
        t0 = time.time()
        q = msg
        if self.history:                                          # reformulate only when there is history
            h = "\n".join(f"{m['role']}: {m['text'][:60]}" for m in self.history[-6:])
            q = client.models.generate_content(model="gemini-2.5-flash",
                contents=f"Rewrite as standalone, resolve pronouns.\n{h}\nFollow-up: {msg}\nStandalone:").text.strip()
            self.stats["reformulated"] += 1
        qe = embed(q)                                             # retrieve on the standalone query
        top = sorted(range(len(self.chunks)), key=lambda i: -float(np.dot(qe, self.embs[i])))[:3]
        ctx = "\n".join(self.chunks[i]["text"] for i in top)
        hist = "\n".join(f"{m['role']}: {m['text'][:60]}" for m in self.history[-4:])
        ans = client.models.generate_content(model="gemini-2.5-flash",
            contents=f"{self.system}\n\nContext:\n{ctx}\n\n{hist}\nUser: {msg}\nAssistant:").text.strip()
        self.history += [{"role":"user","text":msg}, {"role":"assistant","text":ans}]
        self.stats["turns"] += 1
        return {"answer":ans, "query":q,
                "sources":[self.chunks[i].get("source","?") for i in top],
                "latency_ms":(time.time()-t0)*1000}

docs = [{"text":"Netsetos offers GenAI, Data Science, Cloud, Full-Stack courses.","source":"about"},
        {"text":"GenAI course: 14999 rupees. Lifetime access, EMI via Razorpay.","source":"pricing"},
        {"text":"Refund: full 7 days, 50% 8-30 days, none after 30. Email support@netsetos.com.","source":"refund"}]
bot = ProductionChatbot(docs)
for msg in ["What GenAI courses do you have?", "How much is it?", "Can I pay monthly?", "What if I want a refund?"]:
    r = bot.chat(msg)
    print(f"You: {msg}\nBot: {r['answer'][:80]}  [{r['latency_ms']:.0f}ms | {r['sources']}]\n")
print(f"Stats: {bot.stats}")

# Output:
# You: How much is it?
# Bot: The GenAI course is 14,999 rupees ...  [512ms | ['pricing']]
# Stats: {'turns': 4, 'reformulated': 3}

**Explanation**

The step-2 loop plus the minimum production scaffolding: a fixed system prompt, returned sources, and a running stats dict. It is a reference implementation, deliberately missing persistence, rate limits, and guardrails - those are step 8's job.

**How the code works - step by step**
- **`__init__`** - stores the system prompt, embeds the docs once, and initialises a `stats` dict tracking `turns` and `reformulated`.
- **`chat`** - times the turn with `t0`; reformulates only when `self.history` is non-empty (so the first turn pays no rewrite call) and increments `reformulated` when it does.
- Retrieves the top-3 chunks by dot product, prepends `self.system` to the generation prompt, and appends the exchange to history.
- Returns `answer`, the `query` actually retrieved on, the `sources`, and `latency_ms`.
- The final `print(bot.stats)` exposes the honest cost accounting.

**In one line:** the four-move loop wrapped with a system prompt, source attribution, and latency + reformulation stats.

**What you'll see:** Each turn prints its answer with a latency reading and source, e.g. "The GenAI course is 14,999 rupees ... [512ms | ['pricing']]", then a final `Stats: {'turns': 4, 'reformulated': 3}` - the first turn skipped reformulation, the other three each paid one extra LLM call.

## 5 - LangChain & LangGraph: three eras, one recommendation

LangChain is the most common way teams ship conversational RAG, and it has changed twice - deprecated `ConversationalRetrievalChain`, transitional history-aware retriever, and the current recommendation: an agent with a retrieval tool plus **LangGraph checkpointer memory**. This cell is that era-3 pattern; the checkpointer resolves pronouns automatically by saving graph state per `thread_id`.

> **Needs a Gemini API key** and a `retriever` from Lesson 8.1 (set `GOOGLE_API_KEY`).

In [ ]:
# CONVERSATIONAL RAG, THE 2026 WAY - agent + retrieval tool + checkpointer memory
# pip install langchain langgraph langchain-google-genai
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
# production: from langgraph.checkpoint.postgres import PostgresSaver

@tool
def search_courses(query: str) -> str:
    """Search Netsetos course docs. Pass a STANDALONE question (no pronouns)."""
    docs = retriever.invoke(query)                 # your vector store from Lesson 8.1
    return "\n\n".join(d.page_content for d in docs)

checkpointer = InMemorySaver()                      # thread_id -> saved graph state
agent = create_agent(
    model=init_chat_model("google_genai:gemini-2.5-flash"),
    tools=[search_courses],
    system_prompt="Answer only from search_courses results. Cite the source.",
    checkpointer=checkpointer,
)

cfg = {"configurable": {"thread_id": "user-42"}}    # one thread per conversation
for msg in ["What GenAI courses do you have?", "How much is it?", "Can I pay monthly?"]:
    out = agent.invoke({"messages": [{"role":"user", "content": msg}]}, cfg)
    print(f"You: {msg}\nBot: {out['messages'][-1].content[:80]}\n")

# Output:
# You: What GenAI courses do you have?
# Bot: Netsetos offers a GenAI Engineering course ... (source: about)
# You: How much is it?     <- 'it' resolved from thread state; agent re-queries the tool
# Bot: The GenAI course is 14,999 rupees ... (source: pricing)
# You: Can I pay monthly?   <- 'pay' still about the GenAI course
# Bot: Yes - EMI is available via Razorpay ... (source: pricing)

**Explanation**

The 2026 way: instead of hand-writing reformulation, you expose retrieval as a tool and let `create_agent` decide when to call it, while a checkpointer saves the whole conversation state keyed by `thread_id`. Read it as the by-hand loop handed to a framework that adds multi-user isolation and durable state for free.

**How the code works - step by step**
- **`@tool search_courses(query)`** - wraps your vector-store retriever; the docstring instructs the model to pass a standalone, pronoun-free question.
- **`InMemorySaver()`** - the checkpointer that maps `thread_id` to saved graph state (swap `PostgresSaver` in production).
- **`create_agent(model, tools, system_prompt, checkpointer)`** - builds the retrieve-decide loop with memory attached.
- **`cfg = {"configurable": {"thread_id": "user-42"}}`** - one thread per conversation; on each `invoke` the checkpointer loads that thread's state, so "it" resolves from saved history and the agent re-queries the tool on its own.

**In one line:** retrieval-as-tool + `create_agent` + a `thread_id`-keyed checkpointer = automatic memory and multi-user isolation.

**What you'll see:** Three turns where the agent resolves references without any hand-written reformulation: "How much is it?" resolves "it" from thread state and re-queries the tool ("14,999 rupees ... (source: pricing)"), and "Can I pay monthly?" stays about the GenAI course ("EMI is available via Razorpay").

## 6 - LlamaIndex ChatEngine: one parameter, full memory

Where LangGraph gives you an agent to configure, LlamaIndex collapses the whole hand-built chatbot into a single method call. `as_chat_engine(chat_mode="condense_plus_context")` does reformulation AND retrieval AND memory in one line - the fastest path from "I have documents" to "I have a chatbot," and the production-recommended mode among the five available.

> **Needs a Gemini API key** and a `documents` collection (set `GOOGLE_API_KEY`).

In [ ]:
# LLAMAINDEX CHATENGINE - condense_plus_context is the production mode
# pip install llama-index llama-index-llms-google-genai llama-index-embeddings-google-genai
from llama_index.core import VectorStoreIndex, Settings
from llama_index.core.memory import ChatMemoryBuffer
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

Settings.llm = GoogleGenAI(model="gemini-2.5-flash")
Settings.embed_model = GoogleGenAIEmbedding(model_name="gemini-embedding-001")

index = VectorStoreIndex.from_documents(documents)          # your docs
chat = index.as_chat_engine(
    chat_mode="condense_plus_context",                        # reformulate AND retrieve
    memory=ChatMemoryBuffer.from_defaults(token_limit=3000),
    system_prompt="You are a Netsetos support assistant. Answer from context only.",
)

print(chat.chat("What GenAI courses do you have?"))
print(chat.chat("How much is it?"))        # 'it' condensed from memory, then retrieved
chat.reset()                                                # clear memory for a new session

# Output:
# Netsetos offers a GenAI Engineering course covering ...
# The GenAI course is priced at 14,999 rupees ...

**Explanation**

A configuration cell, not a from-scratch build: set the Gemini LLM + embedding once via `Settings`, build an index, then ask it for a chat engine in the production mode. `condense_plus_context` is just your reformulate-then-retrieve loop with a token-limited memory buffer behind one parameter.

**How the code works - step by step**
- **`Settings.llm` / `Settings.embed_model`** - register `gemini-2.5-flash` and `gemini-embedding-001` as the current stack, once, globally.
- **`VectorStoreIndex.from_documents(documents)`** - embeds and indexes your docs.
- **`as_chat_engine(chat_mode="condense_plus_context", memory=..., system_prompt=...)`** - the whole feature: condense the follow-up + history into a standalone question, retrieve context, and answer, with memory included.
- **`ChatMemoryBuffer.from_defaults(token_limit=3000)`** - caps history by tokens, counted backward from the most recent turn.
- **`chat.reset()`** - clears memory to start a fresh session.

**In one line:** register the stack -> index docs -> `as_chat_engine(condense_plus_context)` gives condense + retrieve + memory.

**What you'll see:** Two printed answers where "it" is condensed from memory before retrieval: the first lists the GenAI Engineering course, the second returns "The GenAI course is priced at 14,999 rupees ..." - the step-4 chatbot in two method calls.

## 7 - Six memory architectures & the token budget

Step 3 gave you two memory strategies; production uses six (buffer, window, summary, hybrid, entity, vector) and picks by conversation shape. But whichever you pick, it must fit into a finite context alongside the retrieved documents. This cell builds the token-budget discipline that separates a demo from a system that does not fall over at turn 30 - allocate the window, then truncate by priority on overflow.

In [ ]:
# TOKEN BUDGET - allocate the context window, truncate by priority on overflow
# A rough 4-chars-per-token estimate keeps this dependency-free and deterministic.
def est_tokens(text):
    return len(text) // 4

class ContextBudget:
    def __init__(self, window=8000):
        # reserve a share of the window for each part (an example split for an 8K model)
        self.window = window
        self.response_reserve = int(window * 0.25)   # always leave room to answer

    def fit(self, system, history, retrieved):
        budget = self.window - self.response_reserve
        used = est_tokens(system)                       # system prompt: never dropped
        # history and retrieval compete for what is left
        hist = "\n".join(history)
        while est_tokens(hist) > (budget - used) * 0.4 and len(history) > 2:
            history = history[2:]                          # drop oldest turns first (a real system summarizes them - see step 3)
            hist = "[...earlier turns dropped...]\n" + "\n".join(history)
        used += est_tokens(hist)
        docs = []
        for chunk in retrieved:                          # then trim retrieval to fit
            if used + est_tokens(chunk) > budget: break
            docs.append(chunk); used += est_tokens(chunk)
        return {"system":system, "history":hist, "context":docs,
                "tokens_used":used, "headroom":self.window - used}

b = ContextBudget(window=8000)
plan = b.fit(system="You are a Netsetos support assistant.",
             history=[f"user turn {i}: asked about pricing, refunds, and prerequisites in detail "*9 for i in range(20)],
             retrieved=["retrieved policy document chunk with the relevant answer context text "*45 for _ in range(6)])
print(f"tokens used: {plan['tokens_used']} | headroom: {plan['headroom']} | chunks kept: {len(plan['context'])}")
print("history folded:", plan["history"].startswith("[...earlier"))

# Output:
# tokens used: 5426 | headroom: 2574 | chunks kept: 4
# history folded: True   (20 turns dropped down to 14 to fit; 2 of 6 chunks trimmed)

**Explanation**

A dependency-free, deterministic budgeter - no model call, just a 4-chars-per-token estimate and a priority truncation policy. Its whole point is the value judgement it encodes: never drop the system prompt, always reserve response headroom, drop old history first, sacrifice retrieval breadth last.

**How the code works - step by step**
- **`est_tokens(text)`** - a rough `len // 4` estimate that keeps the demo deterministic and offline.
- **`__init__`** - stores the window size and reserves 25% of it (`response_reserve`) so there is always room to answer.
- **`fit(system, history, retrieved)`** - counts the system prompt first (never dropped), then folds history two turns at a time while it exceeds its ~40% share (a real system would summarize instead of drop), then admits retrieved chunks one by one until the budget would be exceeded.
- Returns the assembled parts plus `tokens_used` and `headroom`.

**In one line:** protect the system prompt and response reserve, fold old history first, trim retrieval last.

**What you'll see:** One summary line - `tokens used: 5426 | headroom: 2574 | chunks kept: 4` - and `history folded: True`: the 20-turn history was folded down to 14 to fit, and only 4 of 6 chunks survived before the response reserve would be eaten.

## 8 - Production: sessions, rate limits, guardrails, isolation

Everything so far assumed one user, one process, infinite budget, and good-faith input. Production assumes none of that. This cell builds two of the load-bearing pieces - per-session state so users never collide, and a token-bucket rate limiter that throttles both bursts and sustained load - the spine you wire behind FastAPI and back with Redis in a real deployment.

In [ ]:
# PRODUCTION SCAFFOLDING - per-session state + a token-bucket rate limiter
# In production: back SessionStore with Redis, the bucket with a Redis Lua script.
import time
from collections import defaultdict

class SessionStore:
    def __init__(self):
        self._data = defaultdict(list)              # session_id -> history; Redis in prod
    def append(self, session_id, role, text):
        self._data[session_id].append({"role":role, "text":text})
    def history(self, session_id):
        return self._data[session_id]              # isolated per user - no cross-tenant bleed

class RateLimiter:
    """Token bucket: `rate` tokens/sec, burst up to `capacity`."""
    def __init__(self, rate=5, capacity=10):
        self.rate, self.capacity = rate, capacity
        self.tokens, self.ts = capacity, time.time()
    def allow(self, cost=1):
        now = time.time()
        self.tokens = min(self.capacity, self.tokens + (now - self.ts) * self.rate)
        self.ts = now
        if self.tokens >= cost:
            self.tokens -= cost
            return True
        return False                              # over budget - return 429 to the caller

store, limiter = SessionStore(), RateLimiter(rate=5, capacity=10)
for i in range(12):
    ok = limiter.allow()
    if ok:
        store.append("user-42", "user", f"message {i}")
    print(f"req {i:2d}: {'served' if ok else 'RATE-LIMITED (429)'}")
print(f"user-42 history length: {len(store.history('user-42'))}")

# Output:
# req  0: served ... req 10: RATE-LIMITED (429)  req 11: RATE-LIMITED (429)
# user-42 history length: 10

**Explanation**

Pure-Python production scaffolding, no model call: a session store keyed per user and a classic token-bucket limiter. Read it as the transport-agnostic pattern - swap the dict for Redis and the bucket for a Redis Lua script and this is a real deployment's request path.

**How the code works - step by step**
- **`SessionStore`** - a `defaultdict(list)` mapping `session_id` to that user's history; `append` and `history` keep tenants isolated (no cross-user bleed).
- **`RateLimiter`** - a token bucket: `rate` tokens/sec, burst up to `capacity`.
- **`allow(cost)`** - refills the bucket by elapsed time * rate (capped at capacity), spends `cost` if available and returns True, else returns False (the caller sends a 429).
- The demo fires 12 requests through a bucket of capacity 10 refilling at 5/sec, appending only served turns to `user-42`'s history.

**In one line:** isolate history per session, and gate traffic with a refilling token bucket.

**What you'll see:** Twelve lines marking each request `served` or `RATE-LIMITED (429)` - the opening burst is served up to capacity, later requests are rejected until the bucket refills - and a final `user-42 history length: 10` showing only served turns were stored.

## 9 - Multi-turn evaluation: what single-turn metrics miss

You cannot ship what you cannot measure, and single-turn RAG metrics (faithfulness, answer relevancy on one Q&A) miss failures that only appear across turns - a conversation can be correct on every individual turn yet drift, repeat retrieval, or cross-contaminate facts over ten. This cell uses DeepEval's `ConversationalTestCase` to score the whole conversation, the discipline you wire into CI.

> **Needs a judge LLM configured for DeepEval** (its metrics call an LLM under the hood).

In [ ]:
# MULTI-TURN RAG EVALUATION - conversation-level metrics single-turn RAG misses
# pip install deepeval
from deepeval.test_case import ConversationalTestCase, Turn
from deepeval.metrics import TurnRelevancyMetric, TurnFaithfulnessMetric
from deepeval import evaluate

test = ConversationalTestCase(turns=[
    Turn(role="user", content="What GenAI courses do you have?"),
    Turn(role="assistant", content="The Netsetos GenAI Engineering course.",
         retrieval_context=["Netsetos offers a GenAI Engineering course."]),
    Turn(role="user", content="How much is it?"),
    Turn(role="assistant", content="It costs 14,999 rupees.",
         retrieval_context=["GenAI course: 14999 rupees, lifetime access."]),   # 'it' resolved
])

evaluate(test_cases=[test],
         metrics=[TurnRelevancyMetric(), TurnFaithfulnessMetric()])

# Output:
# Metrics Summary
#   - Turn Relevancy: 1.00 (passed)   reply stays on-thread across turns
#   - Turn Faithfulness: 1.00 (passed) each answer grounded in its turn's context

**Explanation**

An evaluation harness, not a chatbot call: you hand DeepEval a fixed multi-turn conversation with per-turn retrieval context and it grades the conversation as a unit. The point is that these metrics slide across turns, catching failures per-turn scores cannot.

**How the code works - step by step**
- **`ConversationalTestCase(turns=[...])`** - holds the whole conversation; each assistant `Turn` carries its own `retrieval_context`.
- **`TurnRelevancyMetric`** - slides a window of prior turns to judge whether each reply stays on-thread.
- **`TurnFaithfulnessMetric`** - checks each answer against its own turn's retrieved context (per-turn grounding across the conversation).
- **`evaluate(test_cases, metrics)`** - runs both metrics over the case and prints a summary.

**In one line:** score the movie, not the frames - relevancy across turns and faithfulness per turn.

**What you'll see:** A metrics summary: `Turn Relevancy: 1.00 (passed)` (the reply stays on-thread across turns) and `Turn Faithfulness: 1.00 (passed)` (each answer is grounded in its turn's context).

## 10 - India conversational RAG: Hinglish, multilingual, compliance

An India chatbot faces problems the English-only demos never surface: 250M+ people code-switch between Hindi and English mid-sentence ("refund *kaise* milega?"), Devanagari tokenizes far heavier than English, and banking deployments sit under real regulation. This cell builds the code-switch-aware preprocessing front-end - detect language, (in production) transliterate, and embed everything into one multilingual space.

> **Downloads the BGE-m3 model** (~2GB) on first run; no API key needed - it runs locally.

In [ ]:
# INDIA CONVERSATIONAL RAG - a code-switch-aware preprocessing pipeline
# pip install sentence-transformers  (BGE-m3 handles Hindi + English in one space)
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("BAAI/bge-m3")     # multilingual, 100+ languages

def detect_lang(text):
    # production: fastText LID; stub keys off a Devanagari codepoint range
    return "hi" if any("ऀ" <= c <= "ॿ" for c in text) else "en-or-roman"

def preprocess(turn):
    lang = detect_lang(turn)
    # production: if Romanized Hindi, transliterate to Devanagari with IndicXlit here
    return {"text": turn, "lang": lang, "embedding": embedder.encode(turn)}

for turn in ["What is the refund policy?", "Refund kaise milega?", "रिफंड कैसे मिलेगा?"]:
    r = preprocess(turn)
    print(f"{r['lang']:12s} | dim={len(r['embedding'])} | {turn}")

# Output:
# en-or-roman  | dim=1024 | What is the refund policy?
# en-or-roman  | dim=1024 | Refund kaise milega?
# hi           | dim=1024 | रिफंड कैसे मिलेगा?

**Explanation**

A preprocessing pipeline that normalises language before retrieval. The key idea it demonstrates: a single multilingual embedder (BGE-m3) maps English, Romanized Hinglish, and Devanagari into the same vector space, so all three phrasings of one question can retrieve the same document.

**How the code works - step by step**
- **`SentenceTransformer("BAAI/bge-m3")`** - loads the multilingual embedder (100+ languages) locally.
- **`detect_lang(text)`** - a stub that flags Devanagari by codepoint range (production uses fastText LID).
- **`preprocess(turn)`** - detects language, (production hook: transliterate Romanized Hindi to Devanagari with IndicXlit here), and encodes the turn to a 1024-dim vector.
- The loop runs the same refund question in three scripts - English, Romanized Hinglish, Devanagari - and prints the detected language and embedding dimension.

**In one line:** detect the language, transliterate if needed, and embed all scripts into one shared space.

**What you'll see:** Three lines, one per phrasing, all `dim=1024`: the English and Romanized-Hinglish turns tag as `en-or-roman`, the Devanagari turn tags as `hi` - and because the dimensions match, all three can retrieve the one refund document.

The through-line of every cell: a conversation is state, and each hard problem here is a decision about what state to keep, how to fit it under a finite token budget, and how to prove what you did with it. You built the reformulate -> retrieve -> generate -> remember loop by hand (cells 1-4), then watched LangGraph and LlamaIndex collapse it into an agent or a one-line chat engine, and hardened it with budgets, sessions, rate limits, multi-turn eval, and Indic preprocessing. Next: Lesson 8.10 turns this loop into an agent that decides at runtime whether to retrieve, and 8.11 turns the DeepEval cell into a golden-set CI gate.